In [0]:

# ===================================================
# BLOCK 1 — VALIDATION CONFIGURATION (PYTHON)
# ===================================================

"""
Define the source and governed streaming tables used for post-update
reconciliation and operational acceptance testing.
"""

from pyspark.sql import functions as F

SOURCE_DIRECTORY = (
    "/Volumes/semiconplus_portfolio/landing/external_source/streaming_demo/input"
)

BRONZE_TABLE = "semiconplus_portfolio.bronze.streaming_test_results"
SILVER_TABLE = "semiconplus_portfolio.silver.streaming_test_results"
LATE_TABLE = (
    "semiconplus_portfolio.silver.streaming_late_test_results"
)
QUARANTINE_TABLE = "semiconplus_portfolio.quarantine.streaming_test_results"
GOLD_TABLE = (
    "semiconplus_portfolio.gold.mart_streaming_yield_5m"
)

VALID_EVENT_TYPES = ["TEST_RESULT"]
VALID_STATUSES = ["PASS", "FAIL", "ALARM"]

In [0]:
# ===================================================
# BLOCK 2 — TABLE AVAILABILITY (PYTHON)
# ===================================================

"""
Confirm that the pipeline published every required governed object before
executing row-level and aggregate reconciliation controls.
"""

required_tables = [
    BRONZE_TABLE,
    SILVER_TABLE,
    LATE_TABLE,
    QUARANTINE_TABLE,
    GOLD_TABLE,
]

availability_results = []

for table_name in required_tables:
    table_exists = spark.catalog.tableExists(table_name)
    availability_results.append((table_name, table_exists))
    assert table_exists, f"Required pipeline table does not exist: {table_name}"

display(
    spark.createDataFrame(
        availability_results,
        ["table_name", "table_exists"],
    )
)

In [0]:
# ===================================================
# BLOCK 3 — SOURCE-TO-BRONZE RECONCILIATION (PYTHON)
# ===================================================

"""
Reconcile staged JSON records and distinct source files to Bronze ingestion,
proving that the two triggered updates processed the controlled arrival set.
"""

source_df = spark.read.option("multiLine", "false").json(SOURCE_DIRECTORY)
bronze_df = spark.table(BRONZE_TABLE)

source_record_count = source_df.count()
bronze_record_count = bronze_df.count()
source_file_count = len(
    [
        item
        for item in dbutils.fs.ls(SOURCE_DIRECTORY)
        if not item.isDir() and item.name.lower().endswith(".json")
    ]
)
bronze_file_count = bronze_df.select("_source_file_name").distinct().count()

display(
    spark.createDataFrame(
        [
            (
                source_file_count,
                bronze_file_count,
                source_record_count,
                bronze_record_count,
            )
        ],
        [
            "source_file_count",
            "bronze_file_count",
            "source_record_count",
            "bronze_record_count",
        ],
    )
)

assert source_file_count == bronze_file_count
assert source_record_count == bronze_record_count

In [0]:
# ===================================================
# BLOCK 4 — QUALITY AND DEDUPLICATION RECONCILIATION (PYTHON)
# ===================================================

"""
Verify that accepted and quarantined outcomes are explainable from Bronze and
that no duplicate event identifiers survive in the accepted Silver stream.
"""

silver_df = spark.table(SILVER_TABLE)
late_df = spark.table(LATE_TABLE)
quarantine_df = spark.table(QUARANTINE_TABLE)

silver_count = silver_df.count()
late_count = late_df.count()
quarantine_count = quarantine_df.count()
duplicate_silver_count = (
    silver_df.groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

bronze_distinct_valid_ids = (
    bronze_df
    .filter(
        F.col("event_id").isNotNull()
        & F.col("event_timestamp_utc").isNotNull()
        & F.col("ingestion_timestamp_utc").isNotNull()
        & F.col("is_deliberately_late").isNotNull()
        & F.col("device_id").isNotNull()
        & F.col("product_group_id").isNotNull()
        & F.col("site_id").isNotNull()
        & F.col("equipment_id").isNotNull()
        & F.upper(F.trim("event_type")).isin(VALID_EVENT_TYPES)
        & F.upper(F.trim("status")).isin(VALID_STATUSES)
        & F.expr("try_cast(test_time_seconds AS double) > 0")
        & F.col("_rescued_data").isNull()
    )
    .select(F.trim("event_id").alias("event_id"))
    .distinct()
    .count()
)

display(
    spark.createDataFrame(
        [
            (
                bronze_record_count,
                bronze_distinct_valid_ids,
                silver_count,
                late_count,
                quarantine_count,
                duplicate_silver_count,
            )
        ],
        [
            "bronze_rows",
            "bronze_distinct_valid_event_ids",
            "silver_rows",
            "deliberately_late_rows",
            "quarantine_rows",
            "duplicate_silver_event_ids",
        ],
    )
)

assert duplicate_silver_count == 0
assert silver_count <= bronze_distinct_valid_ids
assert silver_count > 0
assert late_count > 0

duplicate_routed_event_ids = (
    silver_df.select("event_id")
    .intersect(late_df.select("event_id"))
    .count()
)

assert duplicate_routed_event_ids == 0

In [0]:
# ===================================================
# BLOCK 5 — QUARANTINE REASON VALIDATION (PYTHON)
# ===================================================

"""
Confirm that every rejected record contains at least one actionable quality
reason and summarize failure patterns for monitoring and remediation.
"""

unexplained_quarantine_count = quarantine_df.filter(
    F.col("_quality_reasons").isNull()
    | (F.size("_quality_reasons") == 0)
).count()

assert unexplained_quarantine_count == 0

display(
    quarantine_df
    .select(F.explode("_quality_reasons").alias("quality_reason"))
    .groupBy("quality_reason")
    .count()
    .orderBy(F.desc("count"), "quality_reason")
)

In [0]:
# ===================================================
# BLOCK 6 — EVENT-TIME AND WATERMARK READINESS (PYTHON)
# ===================================================

"""
Validate accepted event-time coverage and measure source-defined arrival delay
so the two-hour watermark and explicit late-event route can be defended.
"""

event_time_metrics = silver_df.agg(
    F.min("event_timestamp_utc").alias("minimum_event_timestamp_utc"),
    F.max("event_timestamp_utc").alias("maximum_event_timestamp_utc"),
    F.max(
        F.unix_timestamp("ingestion_timestamp_utc")
        - F.unix_timestamp("event_timestamp_utc")
    ).alias("maximum_observed_ingestion_delay_seconds"),
).first()

late_time_metrics = late_df.agg(
    F.min("arrival_delay_seconds").alias("minimum_late_delay_seconds"),
    F.max("arrival_delay_seconds").alias("maximum_late_delay_seconds"),
).first()

assert event_time_metrics["minimum_event_timestamp_utc"] is not None
assert event_time_metrics["maximum_event_timestamp_utc"] is not None

display(
    spark.createDataFrame(
        [event_time_metrics.asDict()]
    )
)

display(spark.createDataFrame([late_time_metrics.asDict()]))

assert late_time_metrics["minimum_late_delay_seconds"] >= 7_200

print(
    "Watermark interpretation: the two-hour threshold is evaluated relative "
    "to the maximum event time observed by the stream, not wall-clock time."
)

In [0]:
# ===================================================
# BLOCK 7 — GOLD MART VALIDATION (PYTHON)
# ===================================================

"""
Verify key, range, and aggregation invariants in the near-real-time Gold mart
before the table is exposed to operational reporting consumers.
"""

gold_df = spark.table(GOLD_TABLE)

invalid_gold_count = gold_df.filter(
    F.col("window_start_utc").isNull()
    | F.col("window_end_utc").isNull()
    | F.col("site_id").isNull()
    | F.col("equipment_id").isNull()
    | F.col("product_group_id").isNull()
    | F.col("device_id").isNull()
    | (F.col("window_end_utc") <= F.col("window_start_utc"))
    | (F.col("event_count") <= 0)
    | (F.col("pass_count") < 0)
    | (F.col("fail_count") < 0)
    | (F.col("alarm_count") < 0)
    | (F.col("pass_count") + F.col("fail_count") + F.col("alarm_count")
       != F.col("event_count"))
    | (
        F.col("yield_rate").isNotNull()
        & ~F.col("yield_rate").between(0.0, 1.0)
    )
    | (F.col("average_test_time_seconds") <= 0)
    | (F.col("maximum_test_time_seconds") <= 0)
).count()

duplicate_gold_keys = (
    gold_df
    .groupBy(
        "window_start_utc",
        "site_id",
        "equipment_id",
        "product_group_id",
        "device_id",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert invalid_gold_count == 0
assert duplicate_gold_keys == 0
assert gold_df.count() > 0

display(
    gold_df.orderBy(F.desc("window_start_utc")).limit(25)
)

In [0]:
# ===================================================
# BLOCK 8 — SECOND-UPDATE RECOVERY EVIDENCE (PYTHON)
# ===================================================

"""
Confirm that records from both controlled arrival waves are present after the
normal second update, providing data-level evidence of checkpoint recovery.
"""

processed_files = (
    bronze_df
    .groupBy("_source_file_name")
    .count()
    .orderBy("_source_file_name")
)

processed_file_count = processed_files.count()

display(processed_files)

assert processed_file_count == source_file_count
assert processed_file_count >= 10, (
    "Run both five-file arrival waves before final acceptance."
)

In [0]:
# ===================================================
# BLOCK 9 — FINAL RESULT (PYTHON)
# ===================================================

"""
Publish the final bounded-streaming acceptance result and confirm that no
continuous processing remains active after the demonstration.
"""

final_result = {
    "status": "PASSED",
    "source_files": source_file_count,
    "bronze_rows": bronze_record_count,
    "silver_rows": silver_count,
    "deliberately_late_rows": late_count,
    "quarantine_rows": quarantine_count,
    "gold_rows": gold_df.count(),
    "duplicate_silver_event_ids": duplicate_silver_count,
    "duplicate_gold_keys": duplicate_gold_keys,
    "pipeline_mode": "TRIGGERED",
}

display(spark.createDataFrame([final_result]))

print("AUGUST 26 STREAMING ACCEPTANCE: PASSED")
print("Confirm the pipeline status is Stopped before ending the work session.")